# Round-Trip Fidelity Analysis

Measures how faithfully a MEDS dataset survives the conversion to RDF and back:

```
D (MEDS parquet) ──► meds2rdf ──► G (.nt / .nt.gz) ──► reconstruct ──► D' ──► compare D vs D'
```

**No external library needed** beyond `polars`. All functions are defined in this notebook.

**Inputs**
- `MEDS_ROOT` – directory with partitioned `.parquet` files (meds-extract output)
- `RDF_ROOT`  – directory with partitioned `.nt` / `.nt.gz` files (meds2rdf output)

**Sections**
1. Configuration
2. MEDS reader functions
3. RDF reader functions (streaming, no rdflib.Graph in memory)
4. Fidelity metric functions
5. Run analysis
6. Display results
7. Export LaTeX table

In [ ]:
from pathlib import Path
import polars as pl
import os

import joblib
from meds2rdf import MedsRDFConverter, NTriplesSink, Config, MEDSSchema
import shutil


def init_dirs(dataset: str, index: int, root="exports"):
    export_dir = f"{root}/{dataset}"
    outcomes_dir = f"{export_dir}/labels"
    meds_cohort_dir = f"{export_dir}/meds/{index}/MEDS_cohort"
    os.makedirs(export_dir, exist_ok=True)
    os.makedirs(outcomes_dir, exist_ok=True)
    os.makedirs(meds_cohort_dir, exist_ok=True)
    os.makedirs(f"{export_dir}/meds/{index}/MEDS_cohort/metadata", exist_ok=True)

    return export_dir, outcomes_dir, meds_cohort_dir


# def create_meds_cohort(
#     events: pl.DataFrame,
#     orig_dir: str,
#     output_dir: str,
#     columns: list[str] = ["subject_id", "code", "time", "numeric_value", "text_value"],
# ):

#     filtered_codes = events.select("code").unique()
#     pl.read_parquet(f"{orig_dir}/metadata/codes.parquet").join(
#         filtered_codes, on="code", how="inner"
#     ).write_parquet(f"{output_dir}/metadata/codes.parquet")

#     shutil.copy(
#         f"{orig_dir}/metadata/dataset.json", f"{output_dir}/metadata/dataset.json"
#     )

#     os.makedirs(f"{output_dir}/data/train", exist_ok=True)
#     os.makedirs(f"{output_dir}/labels/train", exist_ok=True)
#     events.select(columns).write_parquet(f"{output_dir}/data/train/0.parquet")


def split_events(dt: pl.DataFrame):
    subjects = (
        dt.select("subject_id").unique().sample(fraction=1.0, shuffle=True, seed=1234)
    )

    n = subjects.height

    train_end = int(0.8 * n)
    held_out_end = int(0.9 * n)

    # Assign splits
    subjects = subjects.with_columns(
        pl.when(pl.arange(0, n) < train_end)
        .then(pl.lit("train"))
        .when(pl.arange(0, n) < held_out_end)
        .then(pl.lit("held_out"))
        .otherwise(pl.lit("tuning"))
        .alias("split")
    )

    events = dt.join(subjects, on="subject_id", how="left")

    return (subjects, events)


def create_meds_cohort(
    events: pl.DataFrame,
    orig_dir: str,
    output_dir: str,
    columns: list[str] = ["subject_id", "code", "time", "numeric_value", "text_value"],
):
    split_s, split_e = split_events(events)

    split_s.write_parquet(f"{output_dir}/metadata/subject_splits.parquet")

    filtered_codes = events.select("code").unique()
    pl.read_parquet(f"{orig_dir}/metadata/codes.parquet").join(
        filtered_codes, on="code", how="inner"
    ).write_parquet(f"{output_dir}/metadata/codes.parquet")

    shutil.copy(
        f"{orig_dir}/metadata/dataset.json", f"{output_dir}/metadata/dataset.json"
    )

    for split in ["train", "held_out", "tuning"]:
        df_events = split_e.filter(pl.col("split") == split)
        os.makedirs(f"{output_dir}/data/{split}", exist_ok=True)
        os.makedirs(f"{output_dir}/labels/{split}", exist_ok=True)
        df_events.select(columns).write_parquet(f"{output_dir}/data/{split}/0.parquet")

    return (split_s, split_e)


def convert_subsamples_to_meds(dataset, folds=5, size=500):

    lf = pl.scan_parquet(f"{dataset}/MEDS_cohort/data/**/*.parquet", low_memory=True)

    schema = lf.collect_schema()  # use lf.schema on older polars (<1.0)

    if "text_value" not in schema:
        lf = lf.with_columns(pl.lit(None, dtype=pl.Utf8).alias("text_value"))

    events = lf.select("subject_id", "time", "code", "numeric_value", "text_value")

    patients = (
        events.unique("subject_id")
        .select("subject_id")
        .collect(engine="streaming")
        .sample(n=size * folds, shuffle=True, seed=1234)
    )

    patient_samples = [patients.sample(n=size, shuffle=True) for _ in range(folds)]

    for index, sample in enumerate(patient_samples):
        subsample = events.join(
            sample.select("subject_id").lazy(),
            on="subject_id",
            how="inner",
        ).collect(engine="streaming")

        print(f"TOTAL EVENTS: {len(subsample)}")
        print(f"TOTAL CODES: {len(subsample.group_by('code').len())}")

        export_dir, _, meds_cohort_dir = init_dirs(dataset, index, root="exports")

        create_meds_cohort(
            subsample,
            orig_dir=f"{dataset}/MEDS_cohort",
            output_dir=meds_cohort_dir,
        )

        MedsRDFConverter(meds_cohort_dir).convert(
            sink=NTriplesSink(
                Path(f"{export_dir}/meds_{size}_{index}"),
                gzip_mode=False,
            ),
            cfg=Config(schemas=MEDSSchema.all()),
        )

## 1. Configuration
Edit the paths and MEDS-OWL base IRI here.

In [22]:
# ── Datasets to analyse ──────────────────────────────────────────────────────
# Each entry: (name, meds_parquet_root, rdf_nt_root)
DATASETS = [
    ("NEUROVASC", "exports/NEUROVASC/meds", "exports/NEUROVASC", True),
    # ("MIMIC", "exports/MIMIC/meds", "exports/MIMIC", True),
    # ("eICU", "exports/eICU/meds", "exports/eICU", True),
    # ("NWICU", "exports/NWICU/meds", "exports/NWICU", True),
]

# ── MEDS-OWL base IRI (must match what meds2rdf used when minting IRIs) ──────
MEDS_BASE_IRI = "https://teamheka.github.io/meds-ontology#"

# ── Output CSV ────────────────────────────────────────────────────────────────
OUTPUT_CSV = "round_trip_results.csv"

In [23]:
for dataset, _, _, sampling in DATASETS:
    if sampling:
        convert_subsamples_to_meds(dataset=dataset, size=100)

TOTAL EVENTS: 2450
TOTAL CODES: 79


Processing DatasetMetdata: 1batch [00:00, 17.02batch/s]
Processing Event: 6618batch [00:00, 75572.22batch/s]         
Processing Code: 8batch [00:00, 2784.83batch/s]
Processing SubjectSplit: 100batch [00:00, 37742.32batch/s]
Processing Label: 0batch [00:00, ?batch/s]


TOTAL EVENTS: 2456
TOTAL CODES: 83


Processing DatasetMetdata: 1batch [00:00, 705.87batch/s]
Processing Event: 6629batch [00:00, 74525.88batch/s]         
Processing Code: 8batch [00:00, 2837.34batch/s]
Processing SubjectSplit: 100batch [00:00, 34475.62batch/s]
Processing Label: 0batch [00:00, ?batch/s]


TOTAL EVENTS: 2453
TOTAL CODES: 84


Processing DatasetMetdata: 1batch [00:00, 735.46batch/s]
Processing Event: 6615batch [00:00, 77902.81batch/s]         
Processing Code: 8batch [00:00, 2652.52batch/s]
Processing SubjectSplit: 100batch [00:00, 41250.04batch/s]
Processing Label: 0batch [00:00, ?batch/s]

TOTAL EVENTS: 2446
TOTAL CODES: 87



Processing DatasetMetdata: 1batch [00:00, 667.88batch/s]
Processing Event: 6616batch [00:00, 74435.59batch/s]         
Processing Code: 8batch [00:00, 2226.57batch/s]
Processing SubjectSplit: 100batch [00:00, 20343.91batch/s]
Processing Label: 0batch [00:00, ?batch/s]


TOTAL EVENTS: 2450
TOTAL CODES: 82


Processing DatasetMetdata: 1batch [00:00, 644.09batch/s]
Processing Event: 6614batch [00:00, 78282.72batch/s]         
Processing Code: 8batch [00:00, 2774.47batch/s]
Processing SubjectSplit: 100batch [00:00, 35148.78batch/s]
Processing Label: 0batch [00:00, ?batch/s]


## 2. MEDS reader
Loads partitioned `.parquet` files into Polars DataFrames.

In [14]:
import json
from pathlib import Path
import polars as pl


def load_meds_events(meds_root: str) -> pl.DataFrame:
    """
    Read all .parquet files under <meds_root>/data/ recursively and
    return a single DataFrame with canonical MEDS event columns:
        subject_id, time, code, numeric_value, text_value
    Missing optional columns are added as null columns.
    """
    data_dir = Path(meds_root) / "data"
    files = sorted(data_dir.rglob("*.parquet"))
    if not files:
        raise FileNotFoundError(f"No parquet files found under {data_dir}")

    # Cast to canonical types; add missing optional columns as nulls
    schema = {
        "subject_id": pl.Utf8,
        "time": pl.Datetime("us"),
        "code": pl.Utf8,
        "numeric_value": pl.Float64,
        "text_value": pl.Utf8,
    }

    df = pl.concat([pl.read_parquet(f) for f in files], how="diagonal_relaxed")

    for col, dtype in schema.items():
        if col not in df.columns:
            df = df.with_columns(pl.lit(None).cast(dtype).alias(col))
        else:
            df = df.with_columns(pl.col(col).cast(dtype, strict=False))

    return df.select(list(schema.keys()))


def load_meds_codes(meds_root: str) -> pl.DataFrame | None:
    """Load codes.parquet from <meds_root>/metadata/ if it exists."""
    p1 = Path(meds_root) / "data"
    p2 = Path(meds_root) / "metadata" / "codes.parquet"

    codes = (
        pl.scan_parquet(f"{str(p1)}/**/*.parquet")
        .group_by("code")
        .n_unique()
        .select("code")
        .collect(engine="streaming")
    )

    if p2.exists():
        external_codes = (
            pl.read_parquet(p2)
            .select("parent_codes")
            .explode("parent_codes")
            .rename({"parent_codes": "code"})
        )

        codes = pl.concat([codes, external_codes]).unique()
    return codes


def load_meds_splits(meds_root: str) -> pl.DataFrame | None:
    """Load subject_splits.parquet from <meds_root>/metadata/ if it exists."""
    p = Path(meds_root) / "metadata" / "subject_splits.parquet"
    return pl.read_parquet(p) if p.exists() else None

## 3. RDF reader
Streams `.nt` / `.nt.gz` files without loading an `rdflib.Graph` into memory.
Reconstructs MEDS tables by parsing N-Triples line-by-line.

In [15]:
import gzip
import re
from collections import defaultdict


# ── N-Triples regex parsers ───────────────────────────────────────────────────

_NT_RE = re.compile(
    r"^"
    r"(<[^>]+>|_:\S+)"  # subject
    r"\s+(<[^>]+>)"  # predicate
    r'\s+(<[^>]+>|_:\S+|"(?:[^"\\]|\\.)*"(?:\^\^<[^>]+>|@[a-zA-Z-]+)?)'  # object
    r"\s*\.\s*$"
)
_LIT_RE = re.compile(
    r'^"((?:[^"\\]|\\.)*)"'
    r"(?:\^\^<([^>]+)>)?"
    r"(?:@([a-zA-Z-]+))?"
)


def _strip_iri(s: str) -> str:
    return s[1:-1] if s.startswith("<") and s.endswith(">") else s


def _parse_literal(raw: str) -> tuple[str | None, str | None]:
    """Return (lexical_value, datatype_iri) from a raw N-Triples object string."""
    m = _LIT_RE.match(raw)
    if not m:
        return None, None
    value = m.group(1).replace('\\"', '"').replace("\\\\", "\\").replace("\\n", "\n")
    return value, m.group(2)


def _cast_datetime(val: str | None):
    """Parse an ISO 8601 string to a Polars datetime scalar (microsecond precision)."""
    if val is None:
        return None
    for fmt in ("%Y-%m-%dT%H:%M:%S%.f%Z", "%Y-%m-%dT%H:%M:%S"):
        try:
            return pl.Series([val]).str.to_datetime(
                format=fmt, strict=False, time_unit="us"
            )[0]
        except Exception:
            pass
    return None


def _iter_triples(path: Path):
    """Yield (subject, predicate, object) raw strings from one .nt or .nt.gz file."""
    opener = (
        gzip.open(path, "rt", encoding="utf-8")
        if path.suffix == ".gz"
        else open(path, encoding="utf-8")
    )
    with opener as fh:
        for line in fh:
            line = line.strip()
            if not line or line.startswith("#"):
                continue
            m = _NT_RE.match(line)
            if m:
                yield m.group(1), m.group(2), m.group(3)


# ── Main RDF reconstruction functions ────────────────────────────────────────


def _build_rdf_index(rdf_root: str, meds_base_iri: str = MEDS_BASE_IRI) -> dict:
    """
    Stream all .nt / .nt.gz files under rdf_root and build a lightweight
    index: { node_iri: { predicate_iri: [object, ...] } }
    Also tracks which IRIs belong to each MEDS-OWL class.
    Returns a dict with keys: 'props', 'events', 'subjects', 'codes', 'splits', 'labels'.
    """
    B = meds_base_iri
    RDF_TYPE = "http://www.w3.org/1999/02/22-rdf-syntax-ns#type"
    CLASS_MAP = {
        f"{B}Event": "events",
        f"{B}Subject": "subjects",
        f"{B}Code": "codes",
        f"{B}SubjectSplit": "splits",
        f"{B}SubjectLabel": "labels",
    }

    props = defaultdict(lambda: defaultdict(list))
    classes = {k: set() for k in ["events", "subjects", "codes", "splits", "labels"]}

    files = sorted(Path(rdf_root).rglob("*.nt")) + sorted(
        Path(rdf_root).rglob("*.nt.gz")
    )
    if not files:
        raise FileNotFoundError(f"No .nt / .nt.gz files found under {rdf_root}")

    for f in files:
        for s, p, o in _iter_triples(f):
            s, p = _strip_iri(s), _strip_iri(p)
            if p == RDF_TYPE:
                cls = _strip_iri(o)
                if cls in CLASS_MAP:
                    classes[CLASS_MAP[cls]].add(s)
            else:
                props[s][p].append(o)

    return {"props": props, **classes}


def _first_prop(props: dict, node: str, predicate: str) -> str | None:
    vals = props.get(node, {}).get(predicate, [])
    return vals[0] if vals else None


def reconstruct_events(index: dict, meds_base_iri: str = MEDS_BASE_IRI) -> pl.DataFrame:
    """
    Reconstruct the MEDS events DataFrame from the RDF index.
    Returns a DataFrame with columns: subject_id, time, code, numeric_value, text_value.
    """
    B = meds_base_iri
    props = index["props"]
    rows = []

    for ev in index["events"]:
        subj_iri = _strip_iri(_first_prop(props, ev, f"{B}hasSubject") or "")
        code_iri = _strip_iri(_first_prop(props, ev, f"{B}hasCode") or "")

        sid_raw = _first_prop(props, subj_iri, f"{B}subjectId")
        subject_id = _parse_literal(sid_raw)[0] if sid_raw else None

        # code string: prefer from Code node, fall back to codeString on Event
        cs_raw = _first_prop(props, code_iri, f"{B}codeString") or _first_prop(
            props, ev, f"{B}codeString"
        )
        code = _parse_literal(cs_raw)[0] if cs_raw else None

        time_raw, _ = _parse_literal(_first_prop(props, ev, f"{B}time") or "")
        num_raw, _ = _parse_literal(_first_prop(props, ev, f"{B}numericValue") or "")
        txt_raw, _ = _parse_literal(_first_prop(props, ev, f"{B}textValue") or "")

        rows.append(
            {
                "subject_id": subject_id,
                "time": _cast_datetime(time_raw),
                "code": code,
                "numeric_value": float(num_raw) if num_raw is not None else None,
                "text_value": txt_raw if txt_raw else None,
            }
        )

    schema = {
        "subject_id": pl.Utf8,
        "time": pl.Datetime("us"),
        "code": pl.Utf8,
        "numeric_value": pl.Float64,
        "text_value": pl.Utf8,
    }
    if not rows:
        return pl.DataFrame(schema=schema)
    return pl.DataFrame(rows, schema=schema)

    # if not rows:
    #     return pl.DataFrame(
    #         schema={
    #             "subject_id": pl.Utf8,
    #             "time": pl.Datetime("us"),
    #             "code": pl.Utf8,
    #             "numeric_value": pl.Float64,
    #             "text_value": pl.Utf8,
    #         }
    #     )
    # return pl.DataFrame(rows).with_columns(
    #     [
    #         pl.col("subject_id").cast(pl.Utf8),
    #         pl.col("code").cast(pl.Utf8),
    #         pl.col("numeric_value").cast(pl.Float64),
    #         pl.col("text_value").cast(pl.Utf8),
    #     ]
    # )


def reconstruct_codes(index: dict, meds_base_iri: str = MEDS_BASE_IRI) -> pl.DataFrame:
    """Reconstruct the MEDS code metadata DataFrame from the RDF index."""
    B = meds_base_iri
    props = index["props"]
    rows = []

    for code_iri in index["codes"]:
        cs_raw = _first_prop(props, code_iri, f"{B}codeString")
        desc_raw = _first_prop(props, code_iri, f"{B}codeDescription")
        rows.append(
            {
                "code": _parse_literal(cs_raw)[0] if cs_raw else None,
                "description": _parse_literal(desc_raw)[0] if desc_raw else None,
            }
        )

    if not rows:
        return pl.DataFrame(schema={"code": pl.Utf8, "description": pl.Utf8})
    return pl.DataFrame(rows)


def reconstruct_splits(index: dict, meds_base_iri: str = MEDS_BASE_IRI) -> pl.DataFrame:
    """Reconstruct the MEDS subject splits DataFrame from the RDF index."""
    B = meds_base_iri
    props = index["props"]
    rows = []

    for subj_iri in index["subjects"]:
        sid_raw = _first_prop(props, subj_iri, f"{B}subjectId")
        subject_id = _parse_literal(sid_raw)[0] if sid_raw else None
        split_iris = props.get(subj_iri, {}).get(f"{B}assignedSplit", [])
        for sp_iri in split_iris:
            sp_name = _strip_iri(sp_iri).rsplit("/", 1)[-1].rsplit("#", 1)[-1]
            rows.append({"subject_id": subject_id, "split": sp_name})

    if not rows:
        return pl.DataFrame(schema={"subject_id": pl.Utf8, "split": pl.Utf8})
    return pl.DataFrame(rows)

## 4. Fidelity metric functions

In [27]:
import csv
from dataclasses import dataclass, field, asdict


@dataclass
class FidelityReport:
    dataset_name: str
    # structural
    n_events_original: int = 0
    n_events_reconstructed: int = 0
    structural_completeness: float = 0.0
    n_subjects_original: int = 0
    n_subjects_reconstructed: int = 0
    subject_completeness: float = 0.0
    # per-field exact match rates
    fidelity_subject_id: float = 0.0
    fidelity_code: float = 0.0
    fidelity_time: float = 0.0
    fidelity_numeric_value: float = 0.0
    fidelity_text_value: float = 0.0
    # null rates
    null_rate_orig_time: float = 0.0
    null_rate_orig_numeric_value: float = 0.0
    null_rate_orig_text_value: float = 0.0
    null_rate_recon_time: float = 0.0
    null_rate_recon_numeric_value: float = 0.0
    null_rate_recon_text_value: float = 0.0
    # numeric precision
    numeric_max_abs_error: float = 0.0
    numeric_mean_abs_error: float = 0.0
    numeric_relative_error_p99: float = 0.0
    # temporal precision
    time_exact_match_rate: float = 0.0
    time_max_delta_seconds: float = 0.0
    time_mean_delta_seconds: float = 0.0
    # codes
    n_codes_original: int = 0
    n_codes_reconstructed: int = 0
    code_completeness: float = 0.0
    # splits
    n_splits_original: int = 0
    n_splits_reconstructed: int = 0
    splits_exact_match: float = 0.0
    # known losses — always True by design, documented for the paper
    loss_subject_contiguity: bool = True  # RDF is unordered
    loss_null_semantics: bool = True  # absent triple ≠ explicit null
    loss_temporal_ordering: bool = True  # re-derived via sort
    notes: list = field(default_factory=list)

    def to_dict(self):
        d = asdict(self)
        d["notes"] = "; ".join(d["notes"])
        return d

    def summary(self) -> str:
        lines = [
            f"=== {self.dataset_name} ===",
            f"Events  : {self.n_events_reconstructed:,} / {self.n_events_original:,}  ({self.structural_completeness:.2%})",
            f"Subjects: {self.n_subjects_reconstructed:,} / {self.n_subjects_original:,}  ({self.subject_completeness:.2%})",
            f"Codes   : {self.n_codes_reconstructed:,} / {self.n_codes_original:,}  ({self.code_completeness:.2%})",
            "",
            "Field fidelity (exact-match rate on aligned rows):",
            f"  subject_id    {self.fidelity_subject_id:.4f}",
            f"  code          {self.fidelity_code:.4f}",
            f"  time          {self.fidelity_time:.4f}",
            f"  numeric_value {self.fidelity_numeric_value:.4f}",
            f"  text_value    {self.fidelity_text_value:.4f}",
            "",
            "Numeric precision:",
            f"  max |err|    {self.numeric_max_abs_error:.4g}",
            f"  mean |err|   {self.numeric_mean_abs_error:.4g}",
            f"  p99 rel err  {self.numeric_relative_error_p99:.4g}",
            "",
            "Temporal precision:",
            f"  exact match  {self.time_exact_match_rate:.4f}",
            f"  max Δ (s)    {self.time_max_delta_seconds:.3f}",
            f"  mean Δ (s)   {self.time_mean_delta_seconds:.3f}",
        ]
        if self.notes:
            lines += ["", "Notes:"] + [f"  - {n}" for n in self.notes]
        return "\n".join(lines)


# ── Low-level metric helpers ──────────────────────────────────────────────────


def _exact_match_rate(a: pl.Series, b: pl.Series) -> float:
    """Fraction of positions where a[i] == b[i]; null == null counts as a match."""
    if len(a) == 0:
        return 1.0
    both_null = a.is_null() & b.is_null()
    both_equal = (a == b).fill_null(False)
    return float((both_null | both_equal).mean())


def _numeric_fidelity(orig: pl.Series, recon: pl.Series):
    """Returns (exact_match_rate, max_abs_error, mean_abs_error, p99_rel_error)."""
    mask = orig.is_not_null() & recon.is_not_null()
    if mask.sum() == 0:
        return 1.0, 0.0, 0.0, 0.0
    o, r = orig.filter(mask), recon.filter(mask)
    diff = (o - r).abs()
    rel = diff / (o.abs().clip(lower_bound=1e-10))
    return (
        float((diff < 1e-9).mean() or 0.0),
        float(diff.max() or 0.0),
        float(diff.mean() or 0.0),
        float(rel.quantile(0.99) or 0.0),
    )


def _time_fidelity(orig: pl.Series, recon: pl.Series):
    """Returns (fidelity, exact_match_rate, max_delta_s, mean_delta_s)."""
    both_null = orig.is_null() & recon.is_null()
    both_not_null = orig.is_not_null() & recon.is_not_null()
    if both_not_null.sum() == 0:
        exact = float(both_null.mean())
        return exact, exact, 0.0, 0.0
    o, r = (
        orig.filter(both_not_null).cast(pl.Int64),
        recon.filter(both_not_null).cast(pl.Int64),
    )
    delta_us = (o - r).abs()
    delta_s = delta_us / 1_000_000
    fidelity = float((both_null | (orig == recon).fill_null(False)).mean() or 0.0)
    return (
        fidelity,
        float((delta_us == 0).mean() or 0.0),
        float(delta_s.max() or 0.0),
        float(delta_s.mean() or 0.0),
    )


# ── Main analysis function ────────────────────────────────────────────────────


def compute_fidelity(
    orig_events: pl.DataFrame,
    recon_events: pl.DataFrame,
    dataset_name: str,
    orig_codes: pl.DataFrame | None = None,
    recon_codes: pl.DataFrame | None = None,
    orig_splits: pl.DataFrame | None = None,
    recon_splits: pl.DataFrame | None = None,
) -> FidelityReport:
    """
    Compare original MEDS DataFrames against reconstructed ones.
    Rows are aligned by sorting on (subject_id, time, code).
    Returns a FidelityReport with all metrics populated.
    """
    r = FidelityReport(dataset_name=dataset_name)

    sort_cols = [c for c in ["subject_id", "time", "code"] if c in orig_events.columns]
    orig = orig_events.sort(sort_cols, nulls_last=True)
    recon = recon_events.sort(sort_cols, nulls_last=True)

    r.n_events_original = len(orig)
    r.n_events_reconstructed = len(recon)
    r.structural_completeness = (
        r.n_events_reconstructed / r.n_events_original if r.n_events_original else 0.0
    )
    r.n_subjects_original = orig["subject_id"].n_unique()
    r.n_subjects_reconstructed = recon["subject_id"].n_unique()
    r.subject_completeness = (
        r.n_subjects_reconstructed / r.n_subjects_original
        if r.n_subjects_original
        else 0.0
    )

    for col in ["time", "numeric_value", "text_value"]:
        if col in orig.columns:
            r.__dict__[f"null_rate_orig_{col}"] = float(
                orig[col].is_null().mean() or 0.0
            )
        if col in recon.columns:
            r.__dict__[f"null_rate_recon_{col}"] = float(
                recon[col].is_null().mean() or 0.0
            )

    # Align on the shorter side
    n = min(len(orig), len(recon))
    if n < len(orig):
        r.notes.append(f"Aligned on {n:,} rows (reconstructed has fewer events)")
    o, rc = orig.head(n), recon.head(n)

    if n > 0:
        r.fidelity_subject_id = _exact_match_rate(o["subject_id"], rc["subject_id"])
        r.fidelity_code = _exact_match_rate(o["code"], rc["code"])

        (
            r.fidelity_time,
            r.time_exact_match_rate,
            r.time_max_delta_seconds,
            r.time_mean_delta_seconds,
        ) = _time_fidelity(o["time"], rc["time"])

        (
            r.fidelity_numeric_value,
            r.numeric_max_abs_error,
            r.numeric_mean_abs_error,
            r.numeric_relative_error_p99,
        ) = _numeric_fidelity(o["numeric_value"], rc["numeric_value"])

        r.fidelity_text_value = _exact_match_rate(o["text_value"], rc["text_value"])

    # Codes
    if orig_codes is not None and len(orig_codes) > 0:
        r.n_codes_original = (
            orig_codes["code"].n_unique()
            if "code" in orig_codes.columns
            else len(orig_codes)
        )
        r.n_codes_reconstructed = (
            recon_codes["code"].n_unique()
            if recon_codes is not None and len(recon_codes) > 0
            else 0
        )
        r.code_completeness = (
            r.n_codes_reconstructed / r.n_codes_original if r.n_codes_original else 0.0
        )

    # Splits
    if (
        orig_splits is not None
        and len(orig_splits) > 0
        and recon_splits is not None
        and len(recon_splits) > 0
    ):
        orig_splits = orig_splits.with_columns(
            (pl.col("split") + "Split").alias("split")
        )
        print("ORIG SPLIT", orig_splits)
        print("RECON SPLI", recon_splits)
        r.n_splits_original = len(orig_splits)
        r.n_splits_reconstructed = len(recon_splits)
        orig_set = set(
            zip(orig_splits["subject_id"].to_list(), orig_splits["split"].to_list())
        )
        recon_set = set(
            zip(recon_splits["subject_id"].to_list(), recon_splits["split"].to_list())
        )
        r.splits_exact_match = (
            len(orig_set & recon_set) / len(orig_set) if orig_set else 0.0
        )

    return r


def compute_fidelity_with_join(
    orig_events: pl.DataFrame, recon_events: pl.DataFrame, name: str, **kwargs
) -> FidelityReport:
    """
    Fidelity analysis that handles duplicate (subject_id, time, code) keys
    by splitting into unique-key rows (joined) and duplicate-key rows (sorted alignment).
    """
    key = ["subject_id", "time", "code"]

    # ── Split into unique-key and duplicate-key rows ──────────────────────────
    orig_is_dup = pl.struct(key).is_duplicated()
    recon_is_dup = pl.struct(key).is_duplicated()

    orig_unique = orig_events.filter(~orig_is_dup)
    recon_unique = recon_events.filter(~recon_is_dup)

    orig_dup = orig_events.filter(orig_is_dup)
    recon_dup = recon_events.filter(recon_is_dup)

    print(
        f"  Unique-key rows  : {len(orig_unique):,} orig / {len(recon_unique):,} recon"
    )
    print(f"  Duplicate-key rows: {len(orig_dup):,} orig / {len(recon_dup):,} recon")

    # ── Unique rows: join on key, then compare value columns ──────────────────
    joined = orig_unique.join(
        recon_unique.rename(
            {"numeric_value": "numeric_value_r", "text_value": "text_value_r"}
        ),
        on=key,
        how="inner",
    )

    num_fidelity = _exact_match_rate(joined["numeric_value"], joined["numeric_value_r"])
    txt_fidelity = _exact_match_rate(joined["text_value"], joined["text_value_r"])

    _, num_max, num_mean, num_p99 = _numeric_fidelity(
        joined["numeric_value"], joined["numeric_value_r"]
    )

    print(f"\n  Unique-key rows — numeric_value fidelity : {num_fidelity:.4f}")
    print(f"  Unique-key rows — numeric max |err|      : {num_max:.4g}")
    print(f"  Unique-key rows — text_value fidelity    : {txt_fidelity:.4f}")

    # ── Duplicate rows: sort-align (best effort) ──────────────────────────────
    # For duplicates, position-based alignment after sort is the only option.
    # Results are reported separately and flagged as approximate.
    orig_dup_s = orig_dup.sort(key + ["numeric_value", "text_value"], nulls_last=True)
    recon_dup_s = recon_dup.sort(key + ["numeric_value", "text_value"], nulls_last=True)
    n = min(len(orig_dup_s), len(recon_dup_s))

    dup_num_fidelity = _exact_match_rate(
        orig_dup_s["numeric_value"].head(n), recon_dup_s["numeric_value"].head(n)
    )
    dup_txt_fidelity = _exact_match_rate(
        orig_dup_s["text_value"].head(n), recon_dup_s["text_value"].head(n)
    )

    print(
        f"\n  Duplicate-key rows — numeric_value fidelity (approx): {dup_num_fidelity:.4f}"
    )
    print(
        f"  Duplicate-key rows — text_value fidelity    (approx): {dup_txt_fidelity:.4f}"
    )
    print(
        f"  (alignment on duplicate rows is positional after extended sort — treat as lower bound)"
    )

    # ── Delegate the rest to the original compute_fidelity ────────────────────
    # Use the joined unique rows for the main report metrics
    orig_for_report = joined.select(key + ["numeric_value", "text_value"])
    recon_for_report = joined.select(
        [
            *key,
            pl.col("numeric_value_r").alias("numeric_value"),
            pl.col("text_value_r").alias("text_value"),
        ]
    )

    report = compute_fidelity(orig_for_report, recon_for_report, name, **kwargs)

    report.notes.append(
        f"{len(orig_dup):,} duplicate-key rows ({len(orig_dup) / len(orig_events):.1%} of events) "
        f"excluded from join-based metrics; reported separately above"
    )
    report.notes.append(
        f"Unique-key numeric_value fidelity: {num_fidelity:.4f}, max|err|: {num_max:.4g}"
    )
    report.notes.append(
        f"Duplicate-key numeric_value fidelity (approx): {dup_num_fidelity:.4f}"
    )

    return report


# Fields averaged across samples (mean ± std)
_FLOAT_FIELDS = [
    "structural_completeness",
    "subject_completeness",
    "fidelity_subject_id",
    "fidelity_code",
    "fidelity_time",
    "fidelity_numeric_value",
    "fidelity_text_value",
    "null_rate_orig_time",
    "null_rate_orig_numeric_value",
    "null_rate_orig_text_value",
    "null_rate_recon_time",
    "null_rate_recon_numeric_value",
    "null_rate_recon_text_value",
    "numeric_max_abs_error",
    "numeric_mean_abs_error",
    "numeric_relative_error_p99",
    "time_exact_match_rate",
    "time_max_delta_seconds",
    "time_mean_delta_seconds",
    "code_completeness",
    "splits_exact_match",
]
# Integer fields summed then averaged (reported as mean int)
_INT_FIELDS = [
    "n_events_original",
    "n_events_reconstructed",
    "n_subjects_original",
    "n_subjects_reconstructed",
    "n_codes_original",
    "n_codes_reconstructed",
    "n_splits_original",
    "n_splits_reconstructed",
]


def aggregate_reports(local_reports: list[FidelityReport]) -> FidelityReport:
    """
    Aggregate a list of FidelityReport objects (same dataset, different samples)
    into a single report whose float fields contain the mean across samples.
    Standard deviations are stored in report.notes as a JSON-serialisable dict
    so they are preserved in the CSV and can be used in LaTeX tables.
    """
    import statistics

    if not local_reports:
        raise ValueError("No reports to aggregate")
    if len(local_reports) == 1:
        return local_reports[0]

    name = local_reports[0].dataset_name
    agg = FidelityReport(dataset_name=name)

    # Float fields: mean (and std if n > 1)
    stds = {}
    for field in _FLOAT_FIELDS:
        values = [getattr(r, field) for r in local_reports]
        mean = statistics.mean(values)
        std = statistics.stdev(values) if len(values) > 1 else 0.0
        setattr(agg, field, mean)
        stds[field] = std

    # Integer fields: round(mean)
    for field in _INT_FIELDS:
        values = [getattr(r, field) for r in local_reports]
        setattr(agg, field, round(statistics.mean(values)))

    # Bool fields: True only if all samples agree
    for field in [
        "loss_subject_contiguity",
        "loss_null_semantics",
        "loss_temporal_ordering",
    ]:
        setattr(agg, field, all(getattr(r, field) for r in local_reports))

    # Merge notes and append std summary
    all_notes = []
    for i, r in enumerate(local_reports):
        if r.notes:
            all_notes.append(f"[sample {i}] " + "; ".join(r.notes))
    agg.notes = all_notes
    agg.notes.append(
        f"Aggregated over {len(local_reports)} samples — "
        + ", ".join(f"{k}: ±{v:.4g}" for k, v in stds.items() if v > 0)
    )

    return agg


def stds_from_report(report: FidelityReport) -> dict[str, float]:
    """
    Extract per-field standard deviations from the notes of an aggregated report.
    Returns a dict {field: std} for fields that had non-zero std.
    """
    for note in report.notes:
        if note.startswith("Aggregated over"):
            # Parse "field: ±value" pairs
            result = {}
            parts = note.split(" — ", 1)
            if len(parts) == 2:
                for item in parts[1].split(", "):
                    if ": ±" in item:
                        k, v = item.split(": ±")
                        try:
                            result[k.strip()] = float(v)
                        except ValueError:
                            pass
            return result
    return {}

## 5. Run the analysis

In [28]:
reports = []

SAMPLES = 5
SIZE = 100

for name, meds_root, rdf_root, _ in DATASETS:
    local_report = []
    for sample in range(SAMPLES):
        print(f"\n{'─' * 50}")
        print(f"Dataset : {name}")

        meds_cohort_dir = meds_root + f"/{sample}/MEDS_cohort"
        # 1. Load MEDS parquet
        print("  Loading MEDS parquet …", end=" ")
        orig_events = load_meds_events(meds_cohort_dir)
        orig_codes = load_meds_codes(meds_cohort_dir)
        orig_splits = load_meds_splits(meds_cohort_dir)
        print(f"{len(orig_events):,} events")

        # 2. Stream RDF N-Triples and build index
        print("  Streaming RDF N-Triples …", end=" ")
        index = _build_rdf_index(f"{rdf_root}/meds_{SIZE}_{sample}", MEDS_BASE_IRI)
        print(f"{len(index['events']):,} event nodes")

        # 3. Reconstruct MEDS tables from RDF index
        recon_events = reconstruct_events(index, MEDS_BASE_IRI)
        recon_codes = reconstruct_codes(index, MEDS_BASE_IRI)
        recon_splits = reconstruct_splits(index, MEDS_BASE_IRI)

        # 4. Compute fidelity
        report = compute_fidelity_with_join(
            orig_events,
            recon_events,
            name,
            orig_codes=orig_codes,
            recon_codes=recon_codes,
            orig_splits=orig_splits,
            recon_splits=recon_splits,
        )
        print(report.summary())
        local_report.append(report)

    agg = aggregate_reports(local_report)
    print(f"\n{'═' * 50}")
    print(f"AGGREGATED ({name}, {SAMPLES} samples):")
    print(agg.summary())
    reports.append(agg)


# Save CSV
with open(OUTPUT_CSV, "w", newline="") as fh:
    writer = csv.DictWriter(fh, fieldnames=list(reports[0].to_dict().keys()))
    writer.writeheader()
    for r in reports:
        writer.writerow(r.to_dict())
print(f"\n✓ Results saved to {OUTPUT_CSV}")


──────────────────────────────────────────────────
Dataset : NEUROVASC
  Loading MEDS parquet … 2,450 events
  Streaming RDF N-Triples … 2,450 event nodes
  Unique-key rows  : 2,450 orig / 2,450 recon
  Duplicate-key rows: 0 orig / 0 recon

  Unique-key rows — numeric_value fidelity : 1.0000
  Unique-key rows — numeric max |err|      : 0
  Unique-key rows — text_value fidelity    : 1.0000

  Duplicate-key rows — numeric_value fidelity (approx): 1.0000
  Duplicate-key rows — text_value fidelity    (approx): 1.0000
  (alignment on duplicate rows is positional after extended sort — treat as lower bound)
ORIG SPLIT shape: (100, 2)
┌────────────┬─────────────┐
│ subject_id ┆ split       │
│ ---        ┆ ---         │
│ i64        ┆ str         │
╞════════════╪═════════════╡
│ 8247       ┆ trainSplit  │
│ 6787       ┆ trainSplit  │
│ 9617       ┆ trainSplit  │
│ 45         ┆ trainSplit  │
│ 7009       ┆ trainSplit  │
│ …          ┆ …           │
│ 7723       ┆ tuningSplit │
│ 3697       ┆ t

## 6. Display results

In [10]:
import pandas as pd

# ── Structural completeness ───────────────────────────────────────────────────
pd.DataFrame(
    [
        {
            "Dataset": r.dataset_name,
            "Events (orig)": f"{r.n_events_original:,}",
            "Events (recon)": f"{r.n_events_reconstructed:,}",
            "Structural completeness": f"{r.structural_completeness:.2%}",
            "Subjects (orig)": f"{r.n_subjects_original:,}",
            "Subject completeness": f"{r.subject_completeness:.2%}",
            "Codes completeness": f"{r.code_completeness:.2%}",
        }
        for r in reports
    ]
).set_index("Dataset")

,Events (orig),Events (recon),Structural completeness,Subjects (orig),Subject completeness,Codes completeness
Dataset,,,,,,
MIMIC,"161,356","161,356",100.00%,100,100.00%,99.98%
eICU,"1,373,226","1,373,226",100.00%,100,100.00%,99.96%
NWICU,"157,884","157,884",100.00%,100,100.00%,100.00%


In [11]:
# ── Field fidelity ────────────────────────────────────────────────────────────
pd.DataFrame(
    [
        {
            "Dataset": r.dataset_name,
            "subject_id": f"{r.fidelity_subject_id:.4f}",
            "code": f"{r.fidelity_code:.4f}",
            "time": f"{r.fidelity_time:.4f}",
            "numeric_value": f"{r.fidelity_numeric_value:.4f}",
            "text_value": f"{r.fidelity_text_value:.4f}",
        }
        for r in reports
    ]
).set_index("Dataset")

,subject_id,code,time,numeric_value,text_value
Dataset,,,,,
MIMIC,1.0000,1.0000,1.0000,1.0000,0.9813
eICU,1.0000,1.0000,1.0000,1.0000,0.9998
NWICU,1.0000,1.0000,1.0000,1.0000,0.9998


In [12]:
# ── Numeric and temporal precision ───────────────────────────────────────────
pd.DataFrame(
    [
        {
            "Dataset": r.dataset_name,
            "max |err|": f"{r.numeric_max_abs_error:.2e}",
            "mean |err|": f"{r.numeric_mean_abs_error:.2e}",
            "p99 rel err": f"{r.numeric_relative_error_p99:.2e}",
            "time exact match": f"{r.time_exact_match_rate:.4f}",
            "max Δ time (s)": f"{r.time_max_delta_seconds:.3f}",
            "mean Δ time (s)": f"{r.time_mean_delta_seconds:.3f}",
        }
        for r in reports
    ]
).set_index("Dataset")

,max |err|,mean |err|,p99 rel err,time exact match,max Δ time (s),mean Δ time (s)
Dataset,,,,,,
MIMIC,0.00e+00,0.00e+00,0.00e+00,1.0000,0.000,0.000
eICU,0.00e+00,0.00e+00,0.00e+00,1.0000,0.000,0.000
NWICU,0.00e+00,0.00e+00,0.00e+00,1.0000,0.000,0.000


## 7. Export LaTeX table
Copy the output directly into your paper.

In [22]:
def to_latex_table(reports: list) -> str:
    lines = [
        r"\begin{table}[h]",
        r"\centering",
        r"\caption{Round-trip fidelity results. "
        r"Fidelity values are exact-match rates on aligned event rows. "
        r"Known losses (subject contiguity, null semantics, temporal ordering) "
        r"are by design; see Section~\ref{sec:round-trip}.}",
        r"\label{tab:round-trip}",
        r"\begin{tabular}{lrrrrrrc}",
        r"\hline",
        r"\textbf{Dataset} & \textbf{Struct.} & \textbf{subj\_id} "
        r"& \textbf{code} & \textbf{time} & \textbf{num\_val} "
        r"& \textbf{txt\_val} & \textbf{max$|$err$|$} \\\\",
        r"\hline",
    ]
    for r in reports:
        lines.append(
            f"{r.dataset_name} & "
            f"{r.structural_completeness:.2%} & "
            f"{r.fidelity_subject_id:.4f} & "
            f"{r.fidelity_code:.4f} & "
            f"{r.fidelity_time:.4f} & "
            f"{r.fidelity_numeric_value:.4f} & "
            f"{r.fidelity_text_value:.4f} & "
            f"{r.numeric_max_abs_error:.2e} \\\\"
        )
    lines += [r"\hline", r"\end{tabular}", r"\end{table}"]
    return "\n".join(lines)


print(to_latex_table(reports))

\begin{table}[h]
\centering
\caption{Round-trip fidelity results. Fidelity values are exact-match rates on aligned event rows. Known losses (subject contiguity, null semantics, temporal ordering) are by design; see Section~\ref{sec:round-trip}.}
\label{tab:round-trip}
\begin{tabular}{lrrrrrrc}
\hline
\textbf{Dataset} & \textbf{Struct.} & \textbf{subj\_id} & \textbf{code} & \textbf{time} & \textbf{num\_val} & \textbf{txt\_val} & \textbf{max$|$err$|$} \\\\
\hline
NWICU & 100.00% & 1.0000 & 1.0000 & 1.0000 & 1.0000 & 1.0000 & 0.00e+00 \\
\hline
\end{tabular}
\end{table}
